# Reproduction

一键复现当前实验结果。每个实验单独一个代码框；运行后会打印 `results_summary.json`，并随机抽取一条序列画预测图。

TimesFM 的 direct / LoRA 实验只使用本 notebook 当前 `REPRO_ROOT` 下先跑出的 pretrain checkpoint；缺 checkpoint 会直接报错，不会回退旧 output。

基座模型权重不在本仓库：Chronos 使用 `amazon/chronos-t5-base`，TimesFM 使用 `google/timesfm-2.5-200m-pytorch`。权重需存在于 Hugging Face cache，或由脚本联网下载。

必备环境：当前 Python kernel 需要能导入本项目依赖，包括 `torch`、`transformers`、`peft`、`chronos`、`pyarrow`、`huggingface_hub`、`numpy`、`pandas`、`matplotlib`。

In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import random
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / 'src').is_dir() and (path / 'scripts').is_dir() and (path / 'data').is_dir():
            return path
    raise RuntimeError('Cannot find project root from current working directory')

PROJECT_ROOT = find_project_root()
PROJECT_IMPORT_PATH = os.path.relpath(PROJECT_ROOT, Path.cwd())
if PROJECT_IMPORT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_IMPORT_PATH)

PYTHON = sys.executable
REPRO_ROOT = PROJECT_ROOT / 'output' / f"notebook_repro_{datetime.now():%Y%m%d_%H%M%S}"
REPRO_ROOT.mkdir(parents=True, exist_ok=True)

required = ['torch', 'transformers', 'peft', 'chronos', 'pyarrow', 'huggingface_hub', 'numpy', 'pandas', 'matplotlib']
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(f'Missing required packages: {missing}')

BASE_MODELS = {
    'Chronos': {
        'repo_id': 'amazon/chronos-t5-base',
        'required_any': ['pytorch_model.bin', 'model.safetensors', 'model.safetensors.index.json'],
    },
    'TimesFM': {
        'repo_id': 'google/timesfm-2.5-200m-pytorch',
        'required_any': ['model.safetensors'],
    },
}

def hf_cache_roots() -> list[Path]:
    roots = []
    for env_name in ('HUGGINGFACE_HUB_CACHE', 'HF_HUB_CACHE'):
        value = os.environ.get(env_name)
        if value:
            roots.append(Path(value).expanduser())
    hf_home = os.environ.get('HF_HOME')
    if hf_home:
        roots.append(Path(hf_home).expanduser() / 'hub')
    transformers_cache = os.environ.get('TRANSFORMERS_CACHE')
    if transformers_cache:
        roots.append(Path(transformers_cache).expanduser())
    roots.append(Path.home() / '.cache' / 'huggingface' / 'hub')
    deduped = []
    for root in roots:
        root = root.resolve()
        if root not in deduped:
            deduped.append(root)
    return deduped

def find_hf_snapshot(repo_id: str, required_any: list[str]) -> Path | None:
    repo_cache_name = f"models--{repo_id.replace('/', '--')}"
    for root in hf_cache_roots():
        snapshots = root / repo_cache_name / 'snapshots'
        if not snapshots.exists():
            continue
        for snapshot in sorted(snapshots.iterdir(), reverse=True):
            if snapshot.is_dir() and any((snapshot / name).exists() for name in required_any):
                return snapshot
    return None

def show_base_model_status() -> None:
    print('Base model weights:')
    for name, spec in BASE_MODELS.items():
        snapshot = find_hf_snapshot(spec['repo_id'], spec['required_any'])
        if snapshot is None:
            print(f"  {name}: {spec['repo_id']} not found in local HF cache; scripts will try remote download")
        else:
            print(f"  {name}: {spec['repo_id']} cached at {snapshot}")

def p(rel: str) -> str:
    return str(PROJECT_ROOT / rel)

def out(rel: str) -> Path:
    return REPRO_ROOT / rel

def require_checkpoint(local_rel: str, marker: str, producer: str) -> str:
    path = out(local_rel)
    marker_path = path / marker
    if marker_path.exists():
        return str(path)
    raise FileNotFoundError(
        f"Missing checkpoint file: {marker_path.relative_to(PROJECT_ROOT)}. "
        f"Run run_experiment('{producer}') first in this notebook session."
    )

def run_process(args: list[str], run_dir: Path) -> None:
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / 'train.log'
    cmd = [PYTHON, *args]
    print(' '.join(cmd))
    with log_path.open('w') as log:
        proc = subprocess.Popen(
            cmd,
            cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        code = proc.wait()
    if code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {log_path}')

def show_results(run_dir: Path) -> None:
    summaries = sorted(run_dir.rglob('results_summary.json'))
    if not summaries:
        print(f'No results_summary.json under {run_dir}')
        return
    for summary in summaries:
        print(f'\n{summary.relative_to(PROJECT_ROOT)}')
        rows = json.loads(summary.read_text())
        display(pd.DataFrame(rows))

def _to_float_array(x):
    return np.asarray(x, dtype=float).reshape(-1)

def _load_prediction_files(run_dir: Path) -> list[Path]:
    files = []
    for name in ('predictions.npz', 'zeroshot_predictions.npz'):
        files.extend(sorted(run_dir.rglob(name)))
    return files

def plot_predictions(run_dir: Path, title: str, max_panels: int = 4) -> None:
    files = _load_prediction_files(run_dir)
    if not files:
        print(f'No prediction npz under {run_dir}')
        return
    rng = np.random.default_rng()
    chosen = files[:max_panels]
    fig, axes = plt.subplots(len(chosen), 1, figsize=(10, 3.2 * len(chosen)), squeeze=False)
    axes = axes[:, 0]
    for ax, npz_path in zip(axes, chosen):
        z = np.load(npz_path, allow_pickle=True)
        n = len(z['contexts'])
        i = int(rng.integers(0, n))
        ctx = _to_float_array(z['contexts'][i])
        point = _to_float_array(z['point_forecasts'][i])
        actual = _to_float_array(z['actuals'][i]) if 'actuals' in z.files else None
        quants = np.asarray(z['quantile_forecasts'][i], dtype=float) if 'quantile_forecasts' in z.files else None
        ctx_tail = ctx[-min(len(ctx), 96):]
        x_ctx = np.arange(-len(ctx_tail), 0)
        x_pred = np.arange(len(point))
        ax.plot(x_ctx, ctx_tail, label='context', color='#334155')
        if actual is not None:
            ax.plot(x_pred, actual, label='actual', color='#111827', linewidth=2)
        ax.plot(x_pred, point, label='forecast', color='#dc2626', linewidth=2)
        if quants is not None and quants.ndim == 2 and quants.shape[1] >= 2:
            ax.fill_between(x_pred, quants[:, 0], quants[:, -1], color='#fca5a5', alpha=0.35, label='q10-q90')
        ax.axvline(-0.5, color='#94a3b8', linestyle='--', linewidth=1)
        ax.set_title(f"{title} | {npz_path.relative_to(PROJECT_ROOT)} | sample={i}")
        ax.legend(loc='best')
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

def run_chronos_zeroshot(run_dir: Path) -> None:
    import torch
    from chronos import ChronosPipeline
    from src.chronos_finetune import DEVICE, ZERO_SHOT_LABEL, evaluate_model, parse_tsf, write_tsf
    run_dir.mkdir(parents=True, exist_ok=True)
    tsf_path = p('data/extracted/tourism_monthly_dataset.tsf')
    series, freq = parse_tsf(tsf_path)
    n_eval = max(1, int(len(series) * 0.2))
    eval_tsf = run_dir / 'eval_data.tsf'
    write_tsf(series[-n_eval:], freq, str(eval_tsf))
    pipeline = ChronosPipeline.from_pretrained('amazon/chronos-t5-base', device_map=DEVICE, dtype=torch.float32)
    metrics = evaluate_model(
        pipeline,
        str(eval_tsf),
        prediction_length=24,
        n_series=73,
        num_samples=20,
        freq_str=freq,
        seed=42,
        save_dir=str(run_dir),
    )
    (run_dir / 'results_summary.json').write_text(json.dumps([{'Model': ZERO_SHOT_LABEL, **metrics}], indent=2))

def run_experiment(name: str) -> None:
    spec = EXPERIMENTS[name]
    run_dir = out(spec['out'])
    print(f"\n=== {spec['title']} ===")
    print(f"Expected: {spec['expected']}")
    print(f"Canonical: {spec['canonical']}")
    print(f"Output: {run_dir.relative_to(PROJECT_ROOT)}")
    if 'runner' in spec:
        spec['runner'](run_dir)
    else:
        run_process(spec['args'](run_dir), run_dir)
    show_results(run_dir)
    plot_predictions(run_dir, spec['title'])

EXPERIMENTS = {
    'chronos_zeroshot': {
        'title': 'Chronos zero-shot',
        'expected': 'WQL=1.5441, MASE=1.6617',
        'canonical': 'output/chronos_finetune_base_lora_repro_20260606_011352',
        'out': 'chronos_zeroshot',
        'runner': run_chronos_zeroshot,
    },
    'chronos_ce_lora': {
        'title': 'Chronos CE+LoRA',
        'expected': 'WQL=1.2244, MASE=1.3806',
        'canonical': 'output/chronos_finetune_base_lora_repro_20260606_011352/baseline_ce_lora',
        'out': 'chronos_ce_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'baseline_ce_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_bin_mse_lora': {
        'title': 'Chronos bin-MSE+LoRA',
        'expected': 'WQL=1.2651, MASE=1.4176',
        'canonical': 'output/chronos_finetune_base_lora_repro_20260606_011352/exp1a_bin_mse_lora',
        'out': 'chronos_bin_mse_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp1a_bin_mse_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_mse_ce_lora': {
        'title': 'Chronos bin-MSE+CE+LoRA lambda=1e-4',
        'expected': 'WQL=1.2638, MASE=1.4147',
        'canonical': 'output/chronos_exp2_mse_ce_lora_lambda1e4_repro_20260606_013303/exp2_bin_mse_ce_lora',
        'out': 'chronos_mse_ce_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp2_bin_mse_ce_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--ce-lambda', '1e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_mse_ce_lora_diag': {
        'title': 'Chronos bin-MSE+CE+LoRA lambda=5e-4',
        'expected': 'WQL=1.2656, MASE=1.4344',
        'canonical': 'output/chronos_finetune_base_lora_repro_20260606_011352/exp2_bin_mse_ce_lora',
        'out': 'chronos_mse_ce_lora_lambda5e4',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp2_bin_mse_ce_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--ce-lambda', '5e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_w1_lora': {
        'title': 'Chronos bin-W1+LoRA',
        'expected': 'WQL=1.2750, MASE=1.3151',
        'canonical': 'output/chronos_remaining_losses_repro_20260606_014915/exp4_bin_wass1_lora',
        'out': 'chronos_w1_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp4_bin_wass1_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_w2_lora': {
        'title': 'Chronos bin-W2+LoRA',
        'expected': 'WQL=1.3252, MASE=1.3352',
        'canonical': 'output/chronos_remaining_losses_repro_20260606_014915/exp5_bin_wass2_lora',
        'out': 'chronos_w2_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp5_bin_wass2_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_crps_lora': {
        'title': 'Chronos bin-CRPS+LoRA',
        'expected': 'WQL=1.2325, MASE=1.3713',
        'canonical': 'output/chronos_remaining_losses_repro_20260606_014915/exp7_bin_crps_lora',
        'out': 'chronos_crps_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp7_bin_crps_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_ordinal_lora': {
        'title': 'Chronos bin-OrdinalCE+LoRA',
        'expected': 'WQL=1.2357, MASE=1.3765',
        'canonical': 'output/chronos_remaining_losses_repro_20260606_014915/exp8_bin_ordinal_ce_lora',
        'out': 'chronos_ordinal_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp8_bin_ordinal_ce_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'chronos_huber16_lora': {
        'title': 'Chronos bin-Huber(16)+LoRA',
        'expected': 'WQL=1.2562, MASE=1.3803',
        'canonical': 'output/chronos_remaining_losses_repro_20260606_014915/exp_6a_bin_huber_16_lora',
        'out': 'chronos_huber16_lora',
        'args': lambda d: ['scripts/chronos_finetune.py', '--output-dir', str(d), '--model-id', 'amazon/chronos-t5-base', '--experiments', 'exp_6a_bin_huber_16_lora', '--skip-zeroshot', '--max-steps', '200', '--batch-size', '8', '--grad-accum', '2', '--lora-lr', '3e-4', '--huber-delta-bins', '16', '--prediction-length', '24', '--eval-holdout-ratio', '0.2', '--eval-n-series', '73', '--eval-num-samples', '20', '--seed', '42', '--no-plot'],
    },
    'timesfm_base_zeroshot': {
        'title': 'TimesFM base zero-shot',
        'expected': 'WQL=1.1899, MASE=1.1688',
        'canonical': 'output/timesfm_finetune_repro_20260606_023805',
        'out': 'timesfm_base_zeroshot',
        'args': lambda d: ['scripts/timesfm_baseline.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--prediction-length', '24', '--context-length', '128', '--eval-n-series', '73', '--seed', '42', '--skip-finetune', '--no-plot'],
    },
    'timesfm_base_lora': {
        'title': 'TimesFM base + LoRA',
        'expected': 'WQL=1.2058, MASE=1.1410',
        'canonical': 'output/timesfm_finetune_repro_20260606_023805',
        'out': 'timesfm_base_lora',
        'args': lambda d: ['scripts/timesfm_baseline.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--lr', '1e-4', '--lora-r', '4', '--lora-alpha', '8', '--num-samples', '5000', '--num-workers', '0', '--eval-n-series', '73', '--seed', '42', '--skip-zeroshot', '--no-plot'],
    },
    'timesfm_original_head_pretrain': {
        'title': 'TimesFM original head pretrain',
        'expected': 'internal WQL=0.9926, MASE=1.2253',
        'canonical': 'output/timesfm_original_head_legacy64_repro_20260606_024642',
        'out': 'timesfm_original_head_legacy64_pretrain',
        'args': lambda d: ['scripts/timesfm_baseline.py', '--tsf-paths', 'data/extracted/m4_monthly_dataset.tsf', 'data/extracted/m4_daily_dataset.tsf', 'data/extracted/weather_dataset.tsf', 'data/extracted/electricity_hourly_dataset.tsf', 'data/extracted/traffic_hourly_dataset.tsf', '--output-dir', str(d), '--mode', 'head', '--reset-forecast-head', '--prediction-length', '24', '--context-length', '128', '--epochs', '5', '--batch-size', '256', '--lr', '1e-3', '--num-samples', '200000', '--num-workers', '0', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '100', '--seed', '42', '--skip-zeroshot', '--no-plot'],
    },
    'timesfm_original_head_eval': {
        'title': 'TimesFM original head direct',
        'expected': 'WQL=1.2795, MASE=1.2389',
        'canonical': 'output/timesfm_original_head_legacy64_eval_repro_20260606_030113',
        'out': 'timesfm_original_head_legacy64_eval',
        'args': lambda d: ['scripts/timesfm_baseline.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--mode', 'head', '--forecast-head-checkpoint', require_checkpoint('timesfm_original_head_legacy64_pretrain', 'forecast_head.pt', 'timesfm_original_head_pretrain'), '--skip-zeroshot', '--skip-finetune', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_original_head_lora': {
        'title': 'TimesFM original head + LoRA',
        'expected': 'WQL=1.2049, MASE=1.1468',
        'canonical': 'output/timesfm_lora_from_original_head_legacy64_repro_20260606_030307',
        'out': 'timesfm_original_head_legacy64_lora',
        'args': lambda d: ['scripts/timesfm_baseline.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--mode', 'lora', '--init-forecast-head-checkpoint', require_checkpoint('timesfm_original_head_legacy64_pretrain', 'forecast_head.pt', 'timesfm_original_head_pretrain'), '--skip-zeroshot', '--epochs', '10', '--batch-size', '32', '--lr', '1e-4', '--lora-r', '4', '--lora-alpha', '8', '--num-samples', '5000', '--num-workers', '0', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce64_pretrain': {
        'title': 'TimesFM CE64 pretrain',
        'expected': 'internal WQL=1.2672, MASE=1.4435',
        'canonical': 'output/timesfm_ce_pretrain_strict64_repro_20260606_030942',
        'out': 'timesfm_ce64_pretrain',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-paths', 'data/extracted/m4_monthly_dataset.tsf', 'data/extracted/m4_daily_dataset.tsf', 'data/extracted/weather_dataset.tsf', 'data/extracted/electricity_hourly_dataset.tsf', 'data/extracted/traffic_hourly_dataset.tsf', '--output-dir', str(d), '--prediction-length', '24', '--context-length', '128', '--epochs', '5', '--batch-size', '256', '--num-samples', '200000', '--n-bins', '64', '--bin-range', '-10', '10', '--mode', 'frozen', '--head-lr', '1e-3', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '100', '--seed', '42', '--skip-zeroshot', '--no-plot'],
    },
    'timesfm_ce64_eval': {
        'title': 'TimesFM CE64 direct',
        'expected': 'WQL=1.4605, MASE=1.3809',
        'canonical': 'output/timesfm_ce_legacy64_eval_repro_20260606_032224',
        'out': 'timesfm_ce64_eval',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-checkpoint', require_checkpoint('timesfm_ce64_pretrain', 'ce_head.pt', 'timesfm_ce64_pretrain'), '--skip-zeroshot', '--skip-finetune', '--mode', 'frozen', '--prediction-length', '24', '--context-length', '128', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce64_lora_low': {
        'title': 'TimesFM CE64 + LoRA low LR',
        'expected': 'WQL=1.2308, MASE=1.1162',
        'canonical': 'output/timesfm_ce_legacy64_lora_repeat_repro_20260606_131025',
        'out': 'timesfm_ce64_lora_low_lr',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-head-checkpoint', require_checkpoint('timesfm_ce64_pretrain', 'ce_head.pt', 'timesfm_ce64_pretrain'), '--skip-zeroshot', '--mode', 'lora', '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--num-samples', '5000', '--n-bins', '64', '--head-lr', '5e-5', '--lora-lr', '5e-5', '--lora-r', '4', '--lora-alpha', '8', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce64_lora_high': {
        'title': 'TimesFM CE64 + LoRA old high LR',
        'expected': 'WQL=1.3459, MASE=1.2697',
        'canonical': 'output/timesfm_ce_legacy64_lora_repro_20260606_032337',
        'out': 'timesfm_ce64_lora_high_lr',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-head-checkpoint', require_checkpoint('timesfm_ce64_pretrain', 'ce_head.pt', 'timesfm_ce64_pretrain'), '--skip-zeroshot', '--mode', 'lora', '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--num-samples', '5000', '--n-bins', '64', '--head-lr', '1e-3', '--lora-lr', '1e-4', '--lora-r', '4', '--lora-alpha', '8', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce256_pretrain': {
        'title': 'TimesFM CE256 pretrain',
        'expected': 'internal WQL=0.6285, MASE=0.7708',
        'canonical': 'output/timesfm_ce_pretrain_256_repro_20260606_033145',
        'out': 'timesfm_ce256_pretrain',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-paths', 'data/extracted/m4_monthly_dataset.tsf', 'data/extracted/m4_daily_dataset.tsf', 'data/extracted/weather_dataset.tsf', 'data/extracted/electricity_hourly_dataset.tsf', 'data/extracted/traffic_hourly_dataset.tsf', 'data/extracted/m3_monthly_dataset.tsf', 'data/extracted/m1_monthly_dataset.tsf', '--output-dir', str(d), '--mode', 'frozen', '--n-bins', '256', '--epochs', '5', '--num-samples', '200000', '--batch-size', '128', '--head-lr', '1e-3', '--prediction-length', '24', '--context-length', '128', '--skip-zeroshot', '--eval-n-series', '100', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce256_eval': {
        'title': 'TimesFM CE256 direct',
        'expected': 'WQL=1.3624, MASE=1.2862',
        'canonical': 'output/timesfm_ce_256_eval_repro_20260606_040458',
        'out': 'timesfm_ce256_eval',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-checkpoint', require_checkpoint('timesfm_ce256_pretrain', 'ce_head.pt', 'timesfm_ce256_pretrain'), '--skip-zeroshot', '--skip-finetune', '--mode', 'frozen', '--prediction-length', '24', '--context-length', '128', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce256_lora_1e5': {
        'title': 'TimesFM CE256 + LoRA head_lr=1e-5',
        'expected': 'WQL=1.2691, MASE=1.2336',
        'canonical': 'output/timesfm_ce_256_lora_head1e5_repro_20260606_125004',
        'out': 'timesfm_ce256_lora_head1e5',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-head-checkpoint', require_checkpoint('timesfm_ce256_pretrain', 'ce_head.pt', 'timesfm_ce256_pretrain'), '--skip-zeroshot', '--mode', 'lora', '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--num-samples', '5000', '--n-bins', '256', '--head-lr', '1e-5', '--lora-lr', '5e-5', '--lora-r', '4', '--lora-alpha', '8', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce256_lora_5e5': {
        'title': 'TimesFM CE256 + LoRA head_lr=5e-5',
        'expected': 'WQL=1.3094, MASE=1.2605',
        'canonical': 'output/timesfm_ce_256_lora_repro_20260606_123909',
        'out': 'timesfm_ce256_lora_head5e5',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-head-checkpoint', require_checkpoint('timesfm_ce256_pretrain', 'ce_head.pt', 'timesfm_ce256_pretrain'), '--skip-zeroshot', '--mode', 'lora', '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--num-samples', '5000', '--n-bins', '256', '--head-lr', '5e-5', '--lora-lr', '5e-5', '--lora-r', '4', '--lora-alpha', '8', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce256_lora_freeze': {
        'title': 'TimesFM CE256 + LoRA freeze head',
        'expected': 'WQL=1.3183, MASE=1.3057',
        'canonical': 'output/timesfm_ce_256_lora_freezehead_repro_20260606_125651',
        'out': 'timesfm_ce256_lora_freeze_head',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-head-checkpoint', require_checkpoint('timesfm_ce256_pretrain', 'ce_head.pt', 'timesfm_ce256_pretrain'), '--skip-zeroshot', '--mode', 'lora', '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--num-samples', '5000', '--n-bins', '256', '--head-lr', '0', '--lora-lr', '5e-5', '--lora-r', '4', '--lora-alpha', '8', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_ce256_lora_high': {
        'title': 'TimesFM CE256 + LoRA old high LR',
        'expected': 'WQL=1.3892, MASE=1.4606',
        'canonical': 'output/timesfm_ce_256_lora_repro_20260606_040529',
        'out': 'timesfm_ce256_lora_high_lr',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-head-checkpoint', require_checkpoint('timesfm_ce256_pretrain', 'ce_head.pt', 'timesfm_ce256_pretrain'), '--skip-zeroshot', '--mode', 'lora', '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--num-samples', '5000', '--n-bins', '256', '--head-lr', '1e-3', '--lora-lr', '1e-4', '--lora-r', '4', '--lora-alpha', '8', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_new64_pretrain': {
        'title': 'TimesFM new64 large staged pretrain',
        'expected': 'dependency checkpoint for new64 direct/LoRA',
        'canonical': 'output/timesfm_ce_staged_pretrain_64bin_repro_20260606_041027/01_large',
        'out': 'timesfm_ce_new64_large_pretrain',
        'args': lambda d: ['scripts/timesfm_ce_staged_pretrain.py', '--output-root', str(d), '--stages', 'large', '--starter-epochs', '2', '--starter-num-samples', '50000', '--starter-batch-size', '128', '--large-epochs', '5', '--large-num-samples', '300000', '--large-batch-size', '128', '--context-length', '128', '--prediction-length', '24', '--n-bins', '64', '--head-lr', '1e-3', '--num-workers', '0', '--seed', '42'],
    },
    'timesfm_new64_eval': {
        'title': 'TimesFM new64 large direct',
        'expected': 'WQL=4.5760, MASE=5.5845',
        'canonical': 'output/timesfm_ce_new64_large_eval_repro_20260606_120835',
        'out': 'timesfm_ce_new64_large_eval',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-checkpoint', require_checkpoint('timesfm_ce_new64_large_pretrain/01_large', 'ce_head.pt', 'timesfm_new64_pretrain'), '--skip-zeroshot', '--skip-finetune', '--mode', 'frozen', '--prediction-length', '24', '--context-length', '128', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
    'timesfm_new64_lora': {
        'title': 'TimesFM new64 large + LoRA',
        'expected': 'WQL=1.4641, MASE=1.3968',
        'canonical': 'output/timesfm_ce_new64_large_lora_repro_20260606_124318',
        'out': 'timesfm_ce_new64_large_lora',
        'args': lambda d: ['scripts/timesfm_ce_finetune.py', '--tsf-path', 'data/extracted/tourism_monthly_dataset.tsf', '--output-dir', str(d), '--ce-head-checkpoint', require_checkpoint('timesfm_ce_new64_large_pretrain/01_large', 'ce_head.pt', 'timesfm_new64_pretrain'), '--skip-zeroshot', '--mode', 'lora', '--prediction-length', '24', '--context-length', '128', '--epochs', '10', '--batch-size', '32', '--num-samples', '5000', '--n-bins', '64', '--head-lr', '5e-5', '--lora-lr', '5e-5', '--lora-r', '4', '--lora-alpha', '8', '--val-ratio', '0.1', '--test-ratio', '0.2', '--eval-n-series', '73', '--seed', '42', '--no-plot'],
    },
}

print(f'PROJECT_ROOT={PROJECT_ROOT}')
print(f'PYTHON={PYTHON}')
print(f'REPRO_ROOT={REPRO_ROOT}')
show_base_model_status()

## Chronos

In [ ]:
run_experiment('chronos_zeroshot')

In [ ]:
run_experiment('chronos_ce_lora')

In [ ]:
run_experiment('chronos_bin_mse_lora')

In [ ]:
run_experiment('chronos_mse_ce_lora')

In [ ]:
run_experiment('chronos_mse_ce_lora_diag')

In [ ]:
run_experiment('chronos_w1_lora')

In [ ]:
run_experiment('chronos_w2_lora')

In [ ]:
run_experiment('chronos_crps_lora')

In [ ]:
run_experiment('chronos_ordinal_lora')

In [ ]:
run_experiment('chronos_huber16_lora')

## TimesFM Base / Original Head

In [ ]:
run_experiment('timesfm_base_zeroshot')

In [ ]:
run_experiment('timesfm_base_lora')

In [ ]:
run_experiment('timesfm_original_head_pretrain')

In [ ]:
run_experiment('timesfm_original_head_eval')

In [ ]:
run_experiment('timesfm_original_head_lora')

## TimesFM CE64

In [ ]:
run_experiment('timesfm_ce64_pretrain')

In [ ]:
run_experiment('timesfm_ce64_eval')

In [ ]:
run_experiment('timesfm_ce64_lora_low')

In [ ]:
run_experiment('timesfm_ce64_lora_high')

## TimesFM CE256

In [ ]:
run_experiment('timesfm_ce256_pretrain')

In [ ]:
run_experiment('timesfm_ce256_eval')

In [ ]:
run_experiment('timesfm_ce256_lora_1e5')

In [ ]:
run_experiment('timesfm_ce256_lora_5e5')

In [ ]:
run_experiment('timesfm_ce256_lora_freeze')

In [ ]:
run_experiment('timesfm_ce256_lora_high')

## TimesFM new64 large

In [ ]:
run_experiment('timesfm_new64_pretrain')

In [ ]:
run_experiment('timesfm_new64_eval')

In [ ]:
run_experiment('timesfm_new64_lora')

## Summary

In [ ]:
summaries = []
for path in sorted(REPRO_ROOT.rglob('results_summary.json')):
    rows = json.loads(path.read_text())
    for row in rows:
        summaries.append({'path': str(path.relative_to(PROJECT_ROOT)), **row})
pd.DataFrame(summaries)